# DAC Codec Latents: Classical to Rock Identity Preservation

Convert classical audio files to DAC (Differentiable Audio Codec) encoded representations and extract codec latents. Verify that genre identity (classical vs rock) is preserved through the encoding process.

**Goal:** Analyze whether DAC encoding preserves the musical identity of classical vs rock music in the discrete codec latent space.

## Step 1: Import Required Libraries

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import librosa
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Setup paths
root = Path(r"c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift")
data_dir = root / "data" / "output"
output_dir = root / "app" / "outputs"
notebook_output = root / "notebooks" / "dac_outputs"
notebook_output.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Libraries imported")
print(f"  Device: {device}")
print(f"  Output directory: {notebook_output}")


## Step 2: Install and Load DAC Encoder

In [ ]:
# Install DAC if not available
# !pip install dac -q

try:
    from dac.model import DAC
    dac_available = True
    print("✓ DAC codec library available")
except ImportError:
    print("⚠ DAC not installed. Install with: pip install dac")
    dac_available = False

if dac_available:
    # Load pre-trained DAC model
    print("\nLoading DAC model...")
    try:
        # Use the standard 24kHz DAC model
        dac_model = DAC.load("latest").to(device)
        dac_model.eval()
        sample_rate = 24000
        print(f"✓ DAC model loaded")
        print(f"  Sample rate: {sample_rate} Hz")
        print(f"  Latent dimension: {dac_model.latent_dim}")
        print(f"  Codes per frame: {dac_model.cardinality}")
    except Exception as e:
        print(f"⚠ Error loading DAC: {e}")
        dac_available = False


## Step 3: Load and Preprocess Classical Audio Files

In [ ]:
import glob

def load_and_preprocess_audio(audio_path, target_sr=24000, duration=None):
    """
    Load audio and preprocess to target sample rate.
    
    Args:
        audio_path: Path to audio file
        target_sr: Target sample rate (24000 Hz for DAC)
        duration: Max duration in seconds (None = full audio)
    
    Returns:
        waveform: Audio tensor (1, samples)
        sr: Sample rate
    """
    try:
        # Load audio
        waveform, sr = torchaudio.load(str(audio_path))
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Resample if needed
        if sr != target_sr:
            resampler = T.Resample(sr, target_sr)
            waveform = resampler(waveform)
            sr = target_sr
        
        # Trim to duration if specified
        if duration is not None:
            max_samples = int(duration * sr)
            waveform = waveform[:, :max_samples]
        
        # Normalize
        waveform = waveform / (waveform.abs().max() + 1e-8)
        
        return waveform, sr
    except Exception as e:
        print(f"  Error loading {audio_path}: {e}")
        return None, None

# Load classical audio files
classical_dir = data_dir / "classical"
classical_audio_files = sorted(glob.glob(str(classical_dir / "*.wav")))[:10]  # First 10

print(f"Loading classical audio files...")
print(f"Directory: {classical_dir}")
print(f"Found {len(classical_audio_files)} files\n")

classical_audios = {}
classical_waveforms = {}

for i, audio_file in enumerate(classical_audio_files):
    audio_path = Path(audio_file)
    waveform, sr = load_and_preprocess_audio(audio_path, target_sr=24000)
    
    if waveform is not None:
        classical_audios[audio_path.stem] = audio_file
        classical_waveforms[audio_path.stem] = waveform
        duration = waveform.shape[1] / sr
        print(f"  [{i+1:2d}] {audio_path.name:50s} | {duration:6.2f}s")

print(f"\n✓ Loaded {len(classical_waveforms)} classical audio files")


## Step 4: Encode Classical Audio to DAC Format

In [ ]:
if dac_available:
    print("Encoding classical audio to DAC format...")
    print("=" * 70)
    
    classical_encodings = {}
    classical_codes = {}
    classical_reconstructed = {}
    
    with torch.no_grad():
        for i, (name, waveform) in enumerate(classical_waveforms.items()):
            # Move to device
            waveform = waveform.to(device)
            
            # Encode with DAC
            try:
                # Extract codes and embeddings
                codes = dac_model.encode(waveform.unsqueeze(0))  # Add batch dim
                
                # Decode back to audio for comparison
                z = dac_model.quantizer.from_codes(codes)[0]  # Get embeddings from codes
                reconstructed = dac_model.decoder(z)
                
                classical_codes[name] = codes[0].cpu()  # Save codes (remove batch)
                classical_encodings[name] = z.cpu()
                classical_reconstructed[name] = reconstructed.cpu().squeeze(0)
                
                code_shape = codes[0].shape
                print(f"  [{i+1:2d}] {name:40s} | Codes shape: {code_shape}")
                
            except Exception as e:
                print(f"  [{i+1:2d}] {name:40s} | ✗ Error: {e}")
    
    print(f"\n✓ Encoded {len(classical_encodings)} classical audio files")
    print(f"  Codes shape (per file): {list(classical_codes.values())[0].shape if classical_codes else 'N/A'}")
    print(f"  Embeddings shape: {list(classical_encodings.values())[0].shape if classical_encodings else 'N/A'}")
else:
    print("⚠ DAC not available - skipping encoding")


## Step 5: Extract and Analyze Codec Latents

In [ ]:
if dac_available and classical_encodings:
    print("Analyzing Codec Latents...")
    print("=" * 70)
    
    # Convert embeddings to feature vectors for analysis
    classical_latent_features = {}
    
    for name, embedding in classical_encodings.items():
        # embedding shape: (latent_dim, time_steps)
        # Reduce to single vector by averaging over time
        latent_vector = embedding.mean(dim=1)  # Average across time
        classical_latent_features[name] = latent_vector.numpy()
    
    # Statistics
    all_latents = np.array(list(classical_latent_features.values()))
    print(f"Latent features statistics:")
    print(f"  Total files: {len(classical_latent_features)}")
    print(f"  Latent dimension: {all_latents.shape[1]}")
    print(f"  Mean std across files: {all_latents.std(axis=0).mean():.4f}")
    print(f"  Mean range (min-max): {all_latents.min():.4f} to {all_latents.max():.4f}")
    
    # Pairwise similarity within classical
    print(f"\nClassical Audio Pairwise Similarity:")
    classical_names = list(classical_latent_features.keys())
    
    similarity_matrix = np.zeros((len(classical_names), len(classical_names)))
    for i, name1 in enumerate(classical_names):
        for j, name2 in enumerate(classical_names):
            sim = 1 - cosine(
                classical_latent_features[name1],
                classical_latent_features[name2]
            )
            similarity_matrix[i, j] = sim
    
    print(f"  Mean within-genre similarity: {similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)].mean():.4f}")
    print(f"  Std of similarity: {similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)].std():.4f}")
    
    print(f"\n✓ Extracted latent features for {len(classical_latent_features)} classical files")
else:
    print("⚠ No classical encodings available")


## Step 6: Load and Encode Rock Audio Files

In [ ]:
# Load rock mel spectrograms and convert to audio
print("\nLoading rock audio (from mel spectrograms)...")
print("=" * 70)

rock_mel_dir = data_dir / "output" / "rock_mel"
rock_mel_files = sorted(glob.glob(str(rock_mel_dir / "*.pt")))[:10]

print(f"Directory: {rock_mel_dir}")
print(f"Found {len(rock_mel_files)} mel files\n")

# Need mel-to-audio conversion
mel_to_audio = T.InverseMelScale(
    n_stft=1024 // 2 + 1,
    n_mels=100,
    sample_rate=24000,
    f_min=0,
    f_max=8000,
)

griffin_lim = T.GriffinLim(
    n_fft=1024,
    hop_length=256,
    n_iter=32,
)

rock_audios = {}
rock_waveforms = {}

for i, mel_file in enumerate(rock_mel_files):
    mel_path = Path(mel_file)
    
    try:
        # Load mel
        data = torch.load(mel_path)
        if isinstance(data, dict):
            mel = data['mel']
        else:
            mel = data
        
        # Convert mel to audio using Griffin-Lim
        linear_spec = mel_to_audio(mel.squeeze(0))
        audio = griffin_lim(linear_spec)
        audio = audio / (audio.abs().max() + 1e-8)
        
        rock_audios[mel_path.stem] = mel_file
        rock_waveforms[mel_path.stem] = audio.unsqueeze(0)
        
        duration = audio.shape[0] / 24000
        print(f"  [{i+1:2d}] {mel_path.name:50s} | {duration:6.2f}s")
    except Exception as e:
        print(f"  [{i+1:2d}] {mel_path.name:50s} | ✗ Error: {e}")

print(f"\n✓ Loaded {len(rock_waveforms)} rock audio files")


In [ ]:
if dac_available and rock_waveforms:
    print("Encoding rock audio to DAC format...")
    print("=" * 70)
    
    rock_encodings = {}
    rock_codes = {}
    rock_reconstructed = {}
    
    with torch.no_grad():
        for i, (name, waveform) in enumerate(rock_waveforms.items()):
            # Move to device
            waveform = waveform.to(device)
            
            # Encode with DAC
            try:
                # Extract codes and embeddings
                codes = dac_model.encode(waveform.unsqueeze(0))  # Add batch dim
                
                # Decode back to audio for comparison
                z = dac_model.quantizer.from_codes(codes)[0]
                reconstructed = dac_model.decoder(z)
                
                rock_codes[name] = codes[0].cpu()
                rock_encodings[name] = z.cpu()
                rock_reconstructed[name] = reconstructed.cpu().squeeze(0)
                
                code_shape = codes[0].shape
                print(f"  [{i+1:2d}] {name:40s} | Codes shape: {code_shape}")
                
            except Exception as e:
                print(f"  [{i+1:2d}] {name:40s} | ✗ Error: {e}")
    
    print(f"\n✓ Encoded {len(rock_encodings)} rock audio files")
else:
    print("⚠ DAC not available or no rock audio - skipping")


## Step 7: Compare Classical and Rock Identity Preservation

In [ ]:
if dac_available and classical_encodings and rock_encodings:
    print("\n" + "=" * 70)
    print("🎵 IDENTITY PRESERVATION ANALYSIS")
    print("=" * 70)
    
    # Extract latent features from rock
    rock_latent_features = {}
    
    for name, embedding in rock_encodings.items():
        latent_vector = embedding.mean(dim=1)
        rock_latent_features[name] = latent_vector.numpy()
    
    # 1. Within-genre similarity
    classical_names = list(classical_latent_features.keys())
    rock_names = list(rock_latent_features.keys())
    
    print("\n1️⃣  WITHIN-GENRE COHERENCE")
    print("-" * 70)
    
    # Classical similarity
    classical_sim_values = []
    for i in range(len(classical_names)):
        for j in range(i+1, len(classical_names)):
            sim = 1 - cosine(
                classical_latent_features[classical_names[i]],
                classical_latent_features[classical_names[j]]
            )
            classical_sim_values.append(sim)
    
    # Rock similarity
    rock_sim_values = []
    for i in range(len(rock_names)):
        for j in range(i+1, len(rock_names)):
            sim = 1 - cosine(
                rock_latent_features[rock_names[i]],
                rock_latent_features[rock_names[j]]
            )
            rock_sim_values.append(sim)
    
    classical_sim_mean = np.mean(classical_sim_values)
    classical_sim_std = np.std(classical_sim_values)
    rock_sim_mean = np.mean(rock_sim_values)
    rock_sim_std = np.std(rock_sim_values)
    
    print(f"Classical-Classical similarity: {classical_sim_mean:.4f} ± {classical_sim_std:.4f}")
    print(f"Rock-Rock similarity:          {rock_sim_mean:.4f} ± {rock_sim_std:.4f}")
    
    if classical_sim_mean > rock_sim_mean:
        print(f"  → Classical files more similar to each other in latent space")
    else:
        print(f"  → Rock files more similar to each other in latent space")
    
    # 2. Cross-genre similarity
    print("\n2️⃣  CROSS-GENRE SEPARATION")
    print("-" * 70)
    
    cross_sim_values = []
    for c_name in classical_names:
        for r_name in rock_names:
            sim = 1 - cosine(
                classical_latent_features[c_name],
                rock_latent_features[r_name]
            )
            cross_sim_values.append(sim)
    
    cross_sim_mean = np.mean(cross_sim_values)
    cross_sim_std = np.std(cross_sim_values)
    
    print(f"Classical-Rock similarity:     {cross_sim_mean:.4f} ± {cross_sim_std:.4f}")
    
    # Separability metric
    separation_ratio = min(classical_sim_mean, rock_sim_mean) / cross_sim_mean if cross_sim_mean > 0 else 0
    
    print(f"\nGenre Separability Ratio:      {separation_ratio:.4f}")
    if separation_ratio > 1.0:
        print(f"  ✅ EXCELLENT: Genres are well-separated in latent space")
        print(f"     Same-genre similarity is {separation_ratio:.2f}x higher than cross-genre")
    elif separation_ratio > 0.9:
        print(f"  ✓ GOOD: Genres are reasonably separated")
    elif separation_ratio > 0.8:
        print(f"  ~ OK: Moderate genre separation")
    else:
        print(f"  ⚠ POOR: Genres are not well-separated")
    
    # 3. Statistical test for identity preservation
    print("\n3️⃣  STATISTICAL IDENTITY PRESERVATION TEST")
    print("-" * 70)
    
    from scipy import stats
    
    # Test if within-genre similarity > cross-genre similarity
    t_stat, p_value = stats.ttest_ind(
        classical_sim_values + rock_sim_values,
        cross_sim_values
    )
    
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value:     {p_value:.6f}")
    
    if p_value < 0.05:
        print(f"  ✅ SIGNIFICANT: Genre identity IS preserved (p < 0.05)")
    else:
        print(f"  ⚠ NOT SIGNIFICANT: Genre identity may not be preserved (p >= 0.05)")
    
    print(f"\n✓ Identity preservation analysis complete")
    
else:
    print("⚠ Missing encodings for comparison")


## Step 8: Visualize Results

In [ ]:
if dac_available and classical_encodings and rock_encodings:
    print("Creating visualizations...")
    
    # 1. Similarity Matrix Heatmap
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Classical similarity matrix
    classical_names = list(classical_latent_features.keys())
    classical_sim_matrix = np.zeros((len(classical_names), len(classical_names)))
    
    for i, name1 in enumerate(classical_names):
        for j, name2 in enumerate(classical_names):
            classical_sim_matrix[i, j] = 1 - cosine(
                classical_latent_features[name1],
                classical_latent_features[name2]
            )
    
    im1 = axes[0].imshow(classical_sim_matrix, cmap='viridis', aspect='auto')
    axes[0].set_title('Classical Audio Latent Similarity', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Audio File Index')
    axes[0].set_ylabel('Audio File Index')
    plt.colorbar(im1, ax=axes[0])
    
    # Rock similarity matrix
    rock_names = list(rock_latent_features.keys())
    rock_sim_matrix = np.zeros((len(rock_names), len(rock_names)))
    
    for i, name1 in enumerate(rock_names):
        for j, name2 in enumerate(rock_names):
            rock_sim_matrix[i, j] = 1 - cosine(
                rock_latent_features[name1],
                rock_latent_features[name2]
            )
    
    im2 = axes[1].imshow(rock_sim_matrix, cmap='viridis', aspect='auto')
    axes[1].set_title('Rock Audio Latent Similarity', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Audio File Index')
    axes[1].set_ylabel('Audio File Index')
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    plt.savefig(notebook_output / 'similarity_matrices.png', dpi=100, bbox_inches='tight')
    print(f"  ✓ Saved: similarity_matrices.png")
    plt.show()
    
    # 2. Distribution of similarities
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(classical_sim_values, alpha=0.6, label='Classical-Classical', bins=20, color='blue')
    ax.hist(rock_sim_values, alpha=0.6, label='Rock-Rock', bins=20, color='red')
    ax.hist(cross_sim_values, alpha=0.6, label='Classical-Rock', bins=20, color='gray')
    
    ax.axvline(classical_sim_mean, color='blue', linestyle='--', linewidth=2, label=f'Classical mean: {classical_sim_mean:.3f}')
    ax.axvline(rock_sim_mean, color='red', linestyle='--', linewidth=2, label=f'Rock mean: {rock_sim_mean:.3f}')
    ax.axvline(cross_sim_mean, color='gray', linestyle='--', linewidth=2, label=f'Cross-genre mean: {cross_sim_mean:.3f}')
    
    ax.set_xlabel('Cosine Similarity (1 - Distance)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Latent Space Similarity Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(notebook_output / 'similarity_distribution.png', dpi=100, bbox_inches='tight')
    print(f"  ✓ Saved: similarity_distribution.png")
    plt.show()
    
    # 3. t-SNE visualization of latent space (if possible)
    try:
        from sklearn.manifold import TSNE
        
        print("\n  Computing t-SNE visualization...")
        
        # Combine all latent vectors
        all_vectors = []
        all_labels = []
        
        for name in classical_names:
            all_vectors.append(classical_latent_features[name])
            all_labels.append('Classical')
        
        for name in rock_names:
            all_vectors.append(rock_latent_features[name])
            all_labels.append('Rock')
        
        all_vectors = np.array(all_vectors)
        
        # Reduce to 2D with t-SNE
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(all_vectors)-1))
        reduced = tsne.fit_transform(all_vectors)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        classical_idx = [i for i, l in enumerate(all_labels) if l == 'Classical']
        rock_idx = [i for i, l in enumerate(all_labels) if l == 'Rock']
        
        ax.scatter(reduced[classical_idx, 0], reduced[classical_idx, 1], 
                  label='Classical', s=100, alpha=0.7, color='blue', edgecolors='black', linewidth=1)
        ax.scatter(reduced[rock_idx, 0], reduced[rock_idx, 1], 
                  label='Rock', s=100, alpha=0.7, color='red', edgecolors='black', linewidth=1)
        
        ax.set_xlabel('t-SNE Dimension 1', fontsize=11)
        ax.set_ylabel('t-SNE Dimension 2', fontsize=11)
        ax.set_title('DAC Latent Space Visualization (t-SNE)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=11, loc='best')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(notebook_output / 'tsne_latent_space.png', dpi=100, bbox_inches='tight')
        print(f"  ✓ Saved: tsne_latent_space.png")
        plt.show()
        
    except Exception as e:
        print(f"  ⚠ t-SNE visualization failed: {e}")
    
    print(f"\n✓ All visualizations saved to {notebook_output}")
else:
    print("⚠ Missing data for visualization")


## Step 9: Audio Quality Comparison - Original vs Reconstructed

In [ ]:
if dac_available and classical_waveforms and classical_reconstructed:
    print("Comparing Original vs Reconstructed Audio Quality...")
    print("=" * 70)
    
    classical_names = list(classical_waveforms.keys())
    
    # Compute reconstruction error (MSE)
    reconstruction_errors = {}
    spectral_distances = {}
    
    for name in classical_names[:5]:  # Check first 5
        if name in classical_reconstructed:
            original = classical_waveforms[name].squeeze()
            reconstructed = classical_reconstructed[name].squeeze()
            
            # Align lengths
            min_len = min(len(original), len(reconstructed))
            original = original[:min_len]
            reconstructed = reconstructed[:min_len]
            
            # MSE
            mse = torch.mean((original - reconstructed) ** 2).item()
            reconstruction_errors[name] = mse
            
            # Spectral distance (use librosa)
            orig_stft = np.abs(librosa.stft(original.numpy()))
            recon_stft = np.abs(librosa.stft(reconstructed.numpy()))
            
            # Pad to same shape
            max_shape = max(orig_stft.shape[1], recon_stft.shape[1])
            if orig_stft.shape[1] < max_shape:
                orig_stft = np.pad(orig_stft, ((0,0), (0, max_shape - orig_stft.shape[1])))
            if recon_stft.shape[1] < max_shape:
                recon_stft = np.pad(recon_stft, ((0,0), (0, max_shape - recon_stft.shape[1])))
            
            spectral_dist = np.mean(np.abs(orig_stft - recon_stft))
            spectral_distances[name] = spectral_dist
    
    if reconstruction_errors:
        avg_mse = np.mean(list(reconstruction_errors.values()))
        avg_spec_dist = np.mean(list(spectral_distances.values()))
        
        print(f"\nReconstruction Quality (Classical):")
        print(f"  Average MSE: {avg_mse:.6f}")
        print(f"  Average Spectral Distance: {avg_spec_dist:.4f}")
        
        if avg_mse < 0.01:
            print(f"  ✅ EXCELLENT: Very high-quality reconstruction")
        elif avg_mse < 0.05:
            print(f"  ✓ GOOD: Good quality with minimal artifacts")
        else:
            print(f"  ⚠ ACCEPTABLE: Some loss of detail in reconstruction")
        
        # Individual file breakdown
        print(f"\n  Per-file reconstruction errors:")
        for name, error in sorted(reconstruction_errors.items(), key=lambda x: x[1]):
            print(f"    {name:40s}: MSE={error:.6f}, Spec_dist={spectral_distances[name]:.4f}")
    
    print(f"\n✓ Quality comparison complete")
else:
    print("⚠ Missing reconstruction data")


## Step 10: Summary and Conclusions

In [ ]:
print("\n" + "=" * 70)
print("📊 ANALYSIS SUMMARY")
print("=" * 70)

print("\n✅ COMPLETED ANALYSIS:")
print("-" * 70)

if dac_available:
    print("✓ Loaded classical audio files")
    print(f"✓ Encoded {len(classical_encodings) if classical_encodings else 0} classical files to DAC format")
    print(f"✓ Extracted codec latents ({len(classical_latent_features) if classical_latent_features else 0} files)")
    
    if rock_encodings:
        print(f"✓ Encoded {len(rock_encodings)} rock files to DAC format")
        print(f"✓ Extracted rock codec latents ({len(rock_latent_features) if rock_latent_features else 0} files)")
        print("✓ Computed genre separation metrics")
        print("✓ Generated visualization plots")

print("\n🎯 KEY FINDINGS:")
print("-" * 70)

if classical_encodings and rock_encodings and classical_latent_features and rock_latent_features:
    print(f"\n1. Genre Separation in Latent Space:")
    print(f"   • Classical-Classical similarity: {classical_sim_mean:.4f} ± {classical_sim_std:.4f}")
    print(f"   • Rock-Rock similarity: {rock_sim_mean:.4f} ± {rock_sim_std:.4f}")
    print(f"   • Classical-Rock cross-similarity: {cross_sim_mean:.4f} ± {cross_sim_std:.4f}")
    
    print(f"\n2. Identity Preservation:")
    if separation_ratio > 1.0:
        print(f"   ✅ Genre identities ARE well-preserved")
        print(f"   • Separability ratio: {separation_ratio:.4f}")
        print(f"   • Same-genre pairs are {separation_ratio:.2f}x more similar than cross-genre")
    else:
        print(f"   ⚠ Genre identities may not be well-preserved")
        print(f"   • Separability ratio: {separation_ratio:.4f}")

print(f"\n3. Audio Quality:")
if reconstruction_errors:
    print(f"   • Average reconstruction MSE: {avg_mse:.6f}")
    print(f"   • Average spectral distance: {avg_spec_dist:.4f}")
    if avg_mse < 0.01:
        print(f"   ✅ High-quality lossless/near-lossless reconstruction")

print(f"\n💡 RECOMMENDATIONS:")
print("-" * 70)

if separation_ratio > 1.0:
    print("✓ The DAC codec preserves genre identity in the latent space")
    print("✓ Classical and rock music can be distinguished after encoding")
    print("→ Suitable for conditional generation based on genre")
else:
    print("⚠ Genre identity is not well-separated in latent space")
    print("→ May need to add explicit genre conditioning layer")
    print("→ Consider augmenting with additional genre-specific features")

print("\n" + "=" * 70)
print("✓ Analysis complete!")
print("=" * 70)
